# Notebook 12: Waveform Decomposition & Hemodynamic Features

## Objectives
- Compute Second Derivative PPG (SDPPG) and detect a-e peaks
- Detect fiducial points (systolic peak, dicrotic notch, diastolic peak)
- Extract Windkessel hemodynamic proxies (compliance, resistance, AIx)
- Apply wavelet denoising, EMD, and Hilbert transform
- Visualize waveform decomposition comprehensively


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
from src.config import ARCHIVE_DIR, FEATURES_DIR, ALIGNED_DIR, FIGURES_DIR
from src.data.archive_scanner import scan_archive
from src.data.empatica_loader import load_trial_signals
from scipy import signal


In [ ]:
from src.signals.waveform_decomposition import (
    second_derivative_ppg, detect_sdppg_peaks, detect_fiducial_points,
    compute_augmentation_index, compute_windkessel_features,
    wavelet_denoise, emd_decompose, hilbert_envelope, hilbert_instantaneous_frequency,
    compute_compliance_proxy, compute_resistance_proxy, compute_stiffness_index, compute_pulse_area_ratio
)


In [ ]:
# Load BVP data for waveform analysis
records = scan_archive(ARCHIVE_DIR)
sig_rows = []
for rec in records[:20]:
    try:
        signals, meta = load_trial_signals(rec.files)
        if 'BVP' not in signals:
            continue
        bvp = signals['BVP']['bvp'].to_numpy()
        sr = meta['BVP']['sample_rate']
        # Compute all waveform features
        sdppg = second_derivative_ppg(bvp, sr)
        sdppg_peaks = detect_sdppg_peaks(sdppg, sr)
        fiducial = detect_fiducial_points(bvp, sr)
        windkessel = compute_windkessel_features(bvp, sr)
        row = {
            'subject_id': rec.subject_id,
            'trial_id': rec.trial_id,
            'sdppg_a_count': len(sdppg_peaks.get('a', [])),
            'sdppg_b_count': len(sdppg_peaks.get('b', [])),
            'sdppg_c_count': len(sdppg_peaks.get('c', [])),
            'sdppg_d_count': len(sdppg_peaks.get('d', [])),
            'sdppg_e_count': len(sdppg_peaks.get('e', [])),
        }
        row.update({f'windkessel_{k}': v for k, v in windkessel.items()})
        sig_rows.append(row)
    except Exception as e:
        print(f'Error: {e}')

wf_df = pd.DataFrame(sig_rows) if sig_rows else pd.DataFrame()
print(f'Processed {len(wf_df)} trials')
if len(wf_df) > 0:
    wf_df.head(3)


In [ ]:
# Visualize Windkessel Feature Distributions
if len(wf_df) > 0:
    windkessel_cols = [c for c in wf_df.columns if c.startswith('windkessel_')]
    n = len(windkessel_cols)
    if n > 0:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()
        for i, col in enumerate(windkessel_cols):
            if i >= len(axes):
                break
            vals = wf_df[col].dropna()
            axes[i].hist(vals, bins=15, edgecolor='black', color='coral', alpha=0.7)
            axes[i].set_title(f'{col.replace("windkessel_", "")}\nmean={vals.mean():.2f}')
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'windkessel_features_distributions.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
# Detailed Waveform Decomposition Visualization for a Single Trial
rec = records[0]
signals, meta = load_trial_signals(rec.files)
bvp = signals['BVP']['bvp'].to_numpy()
sr = meta['BVP']['sample_rate']

# Take a 10-second segment
seg_len = int(sr * 10)
segment = bvp[:seg_len] if len(bvp) >= seg_len else bvp

# Compute SDPPG
sdppg = second_derivative_ppg(segment, sr)
sdppg_peaks = detect_sdppg_peaks(sdppg, sr)

# Detect fiducial points
fiducial = detect_fiducial_points(segment, sr)

# Create comprehensive waveform visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
t = np.arange(len(segment)) / sr

# Plot 1: Raw BVP with fiducial points
axes[0].plot(t, segment, 'b-', lw=1.2, label='BVP')
if len(fiducial['systolic_peaks']) > 0:
    sp = fiducial['systolic_peaks']
    axes[0].scatter(t[sp[sp < len(segment)]], segment[sp[sp < len(segment)]],
                   color='red', s=50, zorder=5, label='Systolic Peaks', marker='v')
if len(fiducial['diastolic_notches']) > 0:
    dn = fiducial['diastolic_notches']
    axes[0].scatter(t[dn[dn < len(segment)]], segment[dn[dn < len(segment)]],
                   color='green', s=50, zorder=5, label='Dicrotic Notches', marker='^')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Subject: {rec.subject_id}, Trial: {rec.trial_id} - PPG Waveform with Fiducial Points')
axes[0].legend(), axes[0].grid(True, alpha=0.3)

# Plot 2: SDPPG with a-e peaks
axes[1].plot(t, sdppg, 'purple', lw=1, label='SDPPG (2nd Derivative)')
peak_labels = {'a': 'red', 'b': 'blue', 'c': 'green', 'd': 'orange', 'e': 'brown'}
for label, color in peak_labels.items():
    peaks = sdppg_peaks.get(label, [])
    if len(peaks) > 0 and peaks[0] < len(sdppg):
        axes[1].scatter(t[peaks[0]], sdppg[peaks[0]], color=color, s=80, zorder=5, marker='o')
        axes[1].annotate(label, (t[peaks[0]], sdppg[peaks[0]]),
                        textcoords="offset points", xytext=(5, 5), fontweight='bold', color=color)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('SDPPG with a-e Wave Peaks')
axes[1].legend(), axes[1].grid(True, alpha=0.3)

# Plot 3: Wavelet denoised vs original
try:
    denoised = wavelet_denoise(segment, sr, wavelet='db4', level=4)
    axes[2].plot(t, segment, 'gray', alpha=0.5, lw=0.8, label='Original')
    axes[2].plot(t, denoised, 'darkorange', lw=1.5, label='Wavelet Denoised (db4)')
    axes[2].set_ylabel('Amplitude')
    axes[2].set_title('Wavelet Denoising (Daubechies-4, Level 4)')
    axes[2].legend(), axes[2].grid(True, alpha=0.3)
except Exception as e:
    axes[2].text(0.5, 0.5, f'pywt not available: {e}', ha='center', va='center')

# Plot 4: Hilbert envelope
envelope = hilbert_envelope(segment)
axes[3].plot(t, segment, 'gray', alpha=0.5, lw=0.8, label='Original')
axes[3].plot(t, envelope, 'darkgreen', lw=1.5, label='Hilbert Envelope')
axes[3].set_xlabel('Time (seconds)')
axes[3].set_ylabel('Amplitude')
axes[3].set_title('Hilbert Transform Envelope')
axes[3].legend(), axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'waveform_decomposition_detailed.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Windkessel Hemodynamic Proxy Analysis
if len(wf_df) > 0:
    windkessel_cols = [c for c in wf_df.columns if c.startswith('windkessel_')]
    print('Windkessel Hemodynamic Proxies (all trials):')
    print('=' * 60)
    print(wf_df[['subject_id', 'trial_id'] + windkessel_cols].to_string(index=False))
    print('\nSummary Statistics:')
    print(wf_df[windkessel_cols].describe().round(3))

    # Correlate Windkessel features with existing pseudo-targets
    features = pd.read_csv(FEATURES_DIR / 'trial_features.csv')
    merged = wf_df.merge(features, on=['subject_id', 'trial_id'], how='inner')
    if len(merged) > 0:
        corr_cols = windkessel_cols + ['cv_load_index', 'bp_proxy_score', 'signal_reliability_score']
        corr_matrix = merged[corr_cols].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
                    square=True, ax=ax, cbar_kws={'label': 'Pearson Correlation'})
        ax.set_title('Windkessel Features Correlation with Pseudo-Targets', fontsize=14)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'windkessel_correlation_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
# Augmentation Index (AIx) Analysis
if len(wf_df) > 0:
    aix = wf_df['windkessel_augmentation_index'].dropna()
    print(f'Augmentation Index (AIx = P2/P1):')
    print(f'  Mean: {aix.mean():.3f}')
    print(f'  Std:  {aix.std():.3f}')
    print(f'  Min:  {aix.min():.3f}')
    print(f'  Max:  {aix.max():.3f}')
    print(f'  Range: {aix.max() - aix.min():.3f}')
    print('')
    print('Interpretation: AIx > 1 indicates dominant late systolic peak')
    print('(higher wave reflection, stiffer arteries)')
    print('AIx < 1 indicates dominant early systolic peak')
    print('(lower wave reflection, more elastic arteries)')


In [ ]:
# SDPPG Peak Detection Summary
if len(wf_df) > 0:
    peak_cols = [c for c in wf_df.columns if c.startswith('sdppg_') and c.endswith('_count')]
    print('SDPPG Peak Detection Summary:')
    print('=' * 50)
    print(f"{'Peak':<10} {'Mean Count':<15} {'Std':<10} {'Detection Rate':<15}")
    print('-' * 50)
    for col in peak_cols:
        vals = wf_df[col]
        label = col.replace('sdppg_', '').replace('_count', '')
        mean_val = vals.mean()
        std_val = vals.std()
        detected = (vals > 0).mean() * 100
        print(f'{label:<10} {mean_val:<15.1f} {std_val:<10.2f} {detected:<14.1f}%')
    print('')
    print('SDPPG peaks correspond to:')
    print('  a: Early systolic (largest positive)')
    print('  b: Early systolic trough')
    print('  c: Late systolic bump')
    print('  d: Late diastolic (dicrotic notch rebound)')
    print('  e: End-diastolic')


In [ ]:
print('Waveform Decomposition analysis complete.')
